
# 00 — Extração de campos XML Pricing com Polars

Este notebook:

1. recebe o diretório raiz do dataset no Azure Blob Storage;
2. descobre automaticamente partições no formato:
   - `dt_ingest=YYYY-MM-DD/`
   - `dt_ingest=YYYY-MM-DD/hr_ingest=HH/`;
3. percorre recursivamente todas as subpastas;
4. lê arquivos `.xml` e, opcionalmente, arquivos `.json` que contenham XML em campos como `XmlEnvio` e `XmlRetorno`;
5. transforma os registros em `polars.DataFrame`;
6. consolida schemas diferentes com `diagonal_relaxed`;
7. adiciona colunas de rastreabilidade e das partições;
8. salva o resultado consolidado em Parquet.

> O XML é interpretado com `xml.etree.ElementTree`. A consolidação, transformação e persistência são feitas com Polars.


## 1. Dependências

In [ ]:

# Em Azure ML, execute apenas se as bibliotecas ainda não estiverem disponíveis.
# %pip install -q polars adlfs azure-identity pyarrow


In [ ]:

from __future__ import annotations

import io
import json
import re
import traceback
import xml.etree.ElementTree as ET

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Iterator, Optional

import polars as pl
import adlfs
from azure.identity import DefaultAzureCredential

pl.Config.set_tbl_cols(100)
pl.Config.set_tbl_rows(30)

print("Polars:", pl.__version__)


## 2. Parâmetros

In [ ]:

@dataclass(frozen=True)
class Config:
    # Storage account sem '.blob.core.windows.net'
    account_name: str = "aiadatascienceprod"

    # Container no Blob Storage
    container: str = "landing"

    # Diretório raiz dentro do container.
    # Não inclua o container novamente.
    root_prefix: str = "scd/xml_powercurve_pricing"

    # Extensões pesquisadas recursivamente
    extensions: tuple[str, ...] = (".xml", ".json")

    # Campos de JSON que podem conter XML.
    # Só são usados para arquivos .json.
    json_xml_fields: tuple[str, ...] = ("XmlEnvio", "XmlRetorno")

    # Caso você conheça o XPath do registro repetido, informe aqui.
    # Exemplos: ".//Item", ".//Registro", ".//Proposta"
    # None = detecção automática.
    record_xpath: Optional[str] = None

    # Limite opcional para testes. None processa tudo.
    max_files: Optional[int] = None

    # Em caso de erro, continua lendo os demais arquivos.
    continue_on_error: bool = True

    # Saída local do compute do Azure ML
    output_dir: Path = Path("./outputs")
    output_filename: str = "xml_powercurve_pricing_consolidado.parquet"

CFG = Config()

CFG.output_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = CFG.output_dir / CFG.output_filename

print(CFG)
print("Saída:", OUTPUT_PATH.resolve())


## 3. Autenticação e conexão com o Azure Blob Storage

In [ ]:

credential = DefaultAzureCredential(
    exclude_interactive_browser_credential=False
)

fs = adlfs.AzureBlobFileSystem(
    account_name=CFG.account_name,
    credential=credential,
)

ROOT_PATH = f"{CFG.container}/{CFG.root_prefix.strip('/')}"

print("Diretório remoto:", ROOT_PATH)
print("Existe:", fs.exists(ROOT_PATH))



### Observação sobre autenticação

No Azure Machine Learning, `DefaultAzureCredential` normalmente tenta, entre outras opções:

- Managed Identity do compute/workspace;
- credenciais do Azure CLI;
- login interativo, quando permitido.

A identidade usada precisa ter permissão de leitura no container, por exemplo a função **Storage Blob Data Reader**.


## 4. Descoberta recursiva dos arquivos

In [ ]:

def normalize_remote_path(path: str) -> str:
    """Remove prefixos que algumas versões do adlfs podem retornar."""
    path = str(path).replace("\\", "/")
    account_prefixes = (
        f"abfs://{CFG.container}@{CFG.account_name}.dfs.core.windows.net/",
        f"abfss://{CFG.container}@{CFG.account_name}.dfs.core.windows.net/",
        f"az://{CFG.container}/",
    )
    for prefix in account_prefixes:
        if path.startswith(prefix):
            return f"{CFG.container}/{path[len(prefix):]}"
    return path.lstrip("/")


def discover_files(
    fs: adlfs.AzureBlobFileSystem,
    root_path: str,
    extensions: Iterable[str],
    max_files: Optional[int] = None,
) -> list[str]:
    """Lista recursivamente arquivos abaixo de root_path."""
    normalized_ext = {ext.lower() if ext.startswith(".") else f".{ext.lower()}"
                      for ext in extensions}

    # glob('**') funciona para os dois layouts:
    # dt_ingest=.../arquivo
    # dt_ingest=.../hr_ingest=.../arquivo
    candidates = fs.glob(f"{root_path}/**")

    files = []
    for candidate in candidates:
        candidate = normalize_remote_path(candidate)

        try:
            is_file = fs.isfile(candidate)
        except Exception:
            # Fallback para backends onde isfile pode não estar disponível.
            is_file = Path(candidate).suffix.lower() in normalized_ext

        if is_file and Path(candidate).suffix.lower() in normalized_ext:
            files.append(candidate)

    files = sorted(set(files))

    if max_files is not None:
        files = files[:max_files]

    return files


remote_files = discover_files(
    fs=fs,
    root_path=ROOT_PATH,
    extensions=CFG.extensions,
    max_files=CFG.max_files,
)

print(f"Arquivos encontrados: {len(remote_files):,}")
for path in remote_files[:20]:
    print(path)

if not remote_files:
    raise FileNotFoundError(
        f"Nenhum arquivo {CFG.extensions} foi encontrado abaixo de {ROOT_PATH!r}."
    )


## 5. Extração das partições a partir do caminho

In [ ]:

PARTITION_PATTERN = re.compile(r"(?:^|/)(?P<key>[^/=]+)=(?P<value>[^/]+)(?=/|$)")


def extract_partitions(path: str) -> dict[str, str]:
    """Extrai qualquer partição no padrão chave=valor do caminho."""
    return {
        match.group("key"): match.group("value")
        for match in PARTITION_PATTERN.finditer(path)
    }


partition_preview = [
    {"arquivo": path, **extract_partitions(path)}
    for path in remote_files[:20]
]

pl.DataFrame(partition_preview)


## 6. Funções genéricas para transformar XML em registros

In [ ]:

def strip_namespace(tag: str) -> str:
    """Remove namespace no formato {namespace}tag."""
    return tag.split("}", 1)[-1] if "}" in tag else tag


def clean_text(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    value = value.strip()
    return value if value else None


def element_to_flat_dict(
    element: ET.Element,
    prefix: str = "",
) -> dict[str, Any]:
    """
    Achata um elemento XML.

    Regras:
    - atributos viram colunas com sufixo '@atributo';
    - nós folha viram colunas;
    - caminhos aninhados são unidos por '__';
    - tags repetidas no mesmo registro são concatenadas com ' | '.
    """
    record: dict[str, Any] = {}

    current_tag = strip_namespace(element.tag)
    current_prefix = f"{prefix}__{current_tag}" if prefix else current_tag

    for attr_name, attr_value in element.attrib.items():
        key = f"{current_prefix}@{strip_namespace(attr_name)}"
        record[key] = clean_text(attr_value)

    children = list(element)

    if not children:
        record[current_prefix] = clean_text(element.text)
        return record

    direct_text = clean_text(element.text)
    if direct_text is not None:
        record[f"{current_prefix}__text"] = direct_text

    for child in children:
        child_dict = element_to_flat_dict(child, current_prefix)
        for key, value in child_dict.items():
            if key not in record or record[key] is None:
                record[key] = value
            elif value is not None:
                record[key] = f"{record[key]} | {value}"

    return record


def choose_record_elements(root: ET.Element) -> list[ET.Element]:
    """
    Detecta automaticamente o nível que representa os registros.

    Estratégia:
    1. usa CFG.record_xpath quando informado;
    2. procura grupos de filhos com a mesma tag;
    3. prioriza o maior grupo repetido;
    4. se nada se repetir, usa os filhos diretos;
    5. em último caso, usa o próprio nó raiz.
    """
    if CFG.record_xpath:
        elements = root.findall(CFG.record_xpath)
        if not elements:
            raise ValueError(
                f"Nenhum elemento encontrado para record_xpath={CFG.record_xpath!r}"
            )
        return elements

    best_group: list[ET.Element] = []

    for parent in root.iter():
        children = list(parent)
        if not children:
            continue

        groups: dict[str, list[ET.Element]] = {}
        for child in children:
            groups.setdefault(strip_namespace(child.tag), []).append(child)

        repeated_groups = [group for group in groups.values() if len(group) > 1]

        for group in repeated_groups:
            if len(group) > len(best_group):
                best_group = group

    if best_group:
        return best_group

    direct_children = list(root)
    if direct_children:
        return direct_children

    return [root]


def xml_bytes_to_records(
    xml_content: bytes | str,
    source_file: str,
    xml_field: Optional[str] = None,
) -> list[dict[str, Any]]:
    """Converte um documento XML em uma lista de registros achatados."""
    if isinstance(xml_content, str):
        xml_content = xml_content.encode("utf-8")

    root = ET.fromstring(xml_content)
    record_elements = choose_record_elements(root)

    partitions = extract_partitions(source_file)

    records: list[dict[str, Any]] = []
    for index, element in enumerate(record_elements):
        row = element_to_flat_dict(element)

        row.update({
            "_source_file": source_file,
            "_source_extension": Path(source_file).suffix.lower(),
            "_xml_field": xml_field,
            "_xml_root_tag": strip_namespace(root.tag),
            "_record_tag": strip_namespace(element.tag),
            "_record_index": index,
            **partitions,
        })
        records.append(row)

    return records


## 7. Leitura de `.xml` e de JSON contendo XML

In [ ]:

def read_remote_bytes(
    fs: adlfs.AzureBlobFileSystem,
    remote_path: str,
) -> bytes:
    with fs.open(remote_path, mode="rb") as file:
        return file.read()


def records_from_remote_file(
    fs: adlfs.AzureBlobFileSystem,
    remote_path: str,
) -> list[dict[str, Any]]:
    suffix = Path(remote_path).suffix.lower()
    content = read_remote_bytes(fs, remote_path)

    if suffix == ".xml":
        return xml_bytes_to_records(
            xml_content=content,
            source_file=remote_path,
        )

    if suffix == ".json":
        payload = json.loads(content.decode("utf-8-sig"))

        if not isinstance(payload, dict):
            raise TypeError(
                f"O JSON precisa ser um objeto/dict. Tipo encontrado: {type(payload).__name__}"
            )

        records: list[dict[str, Any]] = []
        found_fields = []

        for field in CFG.json_xml_fields:
            xml_value = payload.get(field)

            if isinstance(xml_value, str) and xml_value.strip():
                found_fields.append(field)
                records.extend(
                    xml_bytes_to_records(
                        xml_content=xml_value,
                        source_file=remote_path,
                        xml_field=field,
                    )
                )

        if not found_fields:
            raise ValueError(
                "Nenhum campo contendo XML foi encontrado no JSON. "
                f"Campos procurados: {CFG.json_xml_fields}. "
                f"Chaves existentes: {list(payload.keys())[:30]}"
            )

        return records

    raise ValueError(f"Extensão não suportada: {suffix}")


## 8. Teste com um arquivo

In [ ]:

sample_file = remote_files[0]
sample_records = records_from_remote_file(fs, sample_file)

print("Arquivo:", sample_file)
print("Registros extraídos:", len(sample_records))

sample_df = pl.from_dicts(sample_records, infer_schema_length=None)
sample_df.head()


## 9. Processamento de todos os arquivos

In [ ]:

def records_to_polars(records: list[dict[str, Any]]) -> pl.DataFrame:
    if not records:
        return pl.DataFrame()

    # strict=False permite acomodar diferenças simples de tipo entre arquivos.
    return pl.from_dicts(
        records,
        infer_schema_length=None,
        strict=False,
    )


dataframes: list[pl.DataFrame] = []
processing_log: list[dict[str, Any]] = []

for file_number, remote_path in enumerate(remote_files, start=1):
    try:
        records = records_from_remote_file(fs, remote_path)
        file_df = records_to_polars(records)

        if file_df.height > 0:
            dataframes.append(file_df)

        processing_log.append({
            "arquivo": remote_path,
            "status": "sucesso",
            "registros": file_df.height,
            "colunas": file_df.width,
            "erro": None,
        })

    except Exception as exc:
        processing_log.append({
            "arquivo": remote_path,
            "status": "erro",
            "registros": 0,
            "colunas": 0,
            "erro": f"{type(exc).__name__}: {exc}",
        })

        if not CFG.continue_on_error:
            raise

    if file_number % 100 == 0 or file_number == len(remote_files):
        print(f"Processados {file_number:,}/{len(remote_files):,} arquivos")


log_df = pl.from_dicts(processing_log, infer_schema_length=None, strict=False)

print(log_df.group_by("status").agg(
    pl.len().alias("arquivos"),
    pl.col("registros").sum().alias("registros"),
))

log_df.filter(pl.col("status") == "erro").head(20)


## 10. Consolidação em um único DataFrame Polars

In [ ]:

if not dataframes:
    raise RuntimeError(
        "Nenhum DataFrame foi produzido. Consulte log_df para identificar os erros."
    )

# diagonal_relaxed:
# - faz a união de todas as colunas;
# - preenche com null quando a coluna não existe em determinado arquivo;
# - tenta reconciliar tipos compatíveis.
df_consolidado = pl.concat(
    dataframes,
    how="diagonal_relaxed",
    rechunk=True,
)

# Colunas de partição e rastreabilidade primeiro.
priority_columns = [
    column
    for column in (
        "dt_ingest",
        "hr_ingest",
        "_source_file",
        "_source_extension",
        "_xml_field",
        "_xml_root_tag",
        "_record_tag",
        "_record_index",
    )
    if column in df_consolidado.columns
]

remaining_columns = [
    column for column in df_consolidado.columns
    if column not in priority_columns
]

df_consolidado = df_consolidado.select(
    priority_columns + sorted(remaining_columns)
)

print("Shape consolidado:", df_consolidado.shape)
print("Arquivos distintos:", df_consolidado.select(
    pl.col("_source_file").n_unique()
).item())

df_consolidado.head()


## 11. Validações

In [ ]:

summary_exprs = [
    pl.len().alias("linhas"),
    pl.col("_source_file").n_unique().alias("arquivos"),
]

if "dt_ingest" in df_consolidado.columns:
    summary_exprs.append(pl.col("dt_ingest").n_unique().alias("datas_ingestao"))

if "hr_ingest" in df_consolidado.columns:
    summary_exprs.append(pl.col("hr_ingest").n_unique().alias("horas_ingestao"))

df_consolidado.select(summary_exprs)


In [ ]:

partition_group_cols = [
    column for column in ("dt_ingest", "hr_ingest")
    if column in df_consolidado.columns
]

if partition_group_cols:
    partition_summary = (
        df_consolidado
        .group_by(partition_group_cols)
        .agg(
            pl.len().alias("registros"),
            pl.col("_source_file").n_unique().alias("arquivos"),
        )
        .sort(partition_group_cols)
    )
    display(partition_summary)
else:
    print("Nenhuma partição chave=valor foi identificada nos caminhos.")


In [ ]:

schema_df = pl.DataFrame({
    "coluna": df_consolidado.columns,
    "tipo": [str(dtype) for dtype in df_consolidado.dtypes],
})

schema_df


## 12. Persistência do resultado

In [ ]:

df_consolidado.write_parquet(
    OUTPUT_PATH,
    compression="zstd",
    statistics=True,
)

LOG_PATH = CFG.output_dir / "00_extracao_campos_xml_pricing_log.parquet"
SCHEMA_PATH = CFG.output_dir / "00_extracao_campos_xml_pricing_schema.csv"

log_df.write_parquet(LOG_PATH, compression="zstd")
schema_df.write_csv(SCHEMA_PATH)

print("Dataset:", OUTPUT_PATH.resolve())
print("Log:", LOG_PATH.resolve())
print("Schema:", SCHEMA_PATH.resolve())



## 13. Leitura posterior com Polars

```python
df = pl.read_parquet("./outputs/xml_powercurve_pricing_consolidado.parquet")
```

Para processamento lazy:

```python
lf = pl.scan_parquet("./outputs/xml_powercurve_pricing_consolidado.parquet")
```



## 14. Alternativa: gravar o Parquet de volta no Blob Storage

O bloco abaixo é opcional. O Parquet é primeiro criado em memória e depois enviado ao Blob.

Para volumes muito grandes, prefira processar e salvar por partição/lote, em vez de manter todo o dataset em memória.


In [ ]:

# REMOTE_OUTPUT_PATH = (
#     f"{CFG.container}/scd/xml_powercurve_pricing_processed/"
#     "xml_powercurve_pricing_consolidado.parquet"
# )
#
# buffer = io.BytesIO()
# df_consolidado.write_parquet(
#     buffer,
#     compression="zstd",
#     statistics=True,
# )
# buffer.seek(0)
#
# with fs.open(REMOTE_OUTPUT_PATH, mode="wb") as file:
#     file.write(buffer.read())
#
# print("Arquivo gravado em:", REMOTE_OUTPUT_PATH)



## 15. Notas para Databricks

O código de descoberta e leitura com `adlfs` também pode ser usado no Databricks, desde que:

- `polars`, `adlfs`, `azure-identity` e `pyarrow` estejam instalados no cluster;
- a identidade do cluster tenha permissão no Storage Account;
- ou as credenciais sejam disponibilizadas por Service Principal/Managed Identity.

Em ambiente Databricks com ADLS montado ou acessível por Unity Catalog, outra opção é fornecer ao Polars caminhos locais/montados. A lógica de partições permanece a mesma, pois a descoberta é recursiva.
